# JARVIS Dashboard — SQL Production
Containers · LLM Backends · Workflows · Orchestration

In [ ]:
import sqlite3, json, subprocess, datetime
from pathlib import Path
import pandas as pd

DB = Path.home() / 'IA/Core/jarvis/data/jarvis.db'
conn = sqlite3.connect(DB)
conn.row_factory = sqlite3.Row
print(f'DB: {DB} — {DB.stat().st_size/1024:.1f} KB')

## Containers

In [ ]:
df = pd.read_sql('SELECT name, image, status, ports, restart_policy, updated_at FROM containers ORDER BY status', conn)
df['ports'] = df['ports'].apply(lambda x: ', '.join(json.loads(x)) if x else '')
df.style.applymap(lambda v: 'color:green' if v=='running' else ('color:red' if v in ('stopped','exited') else ''), subset=['status'])

## LLM Backends

In [ ]:
df_llm = pd.read_sql('SELECT node, address, status, priority, json_array_length(models) as n_models, vram_used_mb, vram_total_mb, updated_at FROM llm_backends ORDER BY priority', conn)
df_llm['vram_pct'] = (df_llm['vram_used_mb'] / df_llm['vram_total_mb'] * 100).round(1).fillna(0)
df_llm.style.applymap(lambda v: 'color:green' if v=='up' else 'color:red', subset=['status'])

In [ ]:
# Models par node
for row in conn.execute('SELECT node, models FROM llm_backends ORDER BY priority'):
    models = json.loads(row['models'] or '[]')
    print(f"\n[{row['node']}] {len(models)} models:")
    for m in models: print(f'  - {m}')

## Orchestration State

In [ ]:
df_orch = pd.read_sql('SELECT component, state, active_tasks, queue_depth, last_heartbeat FROM orchestration_state ORDER BY component', conn)
df_orch.style.applymap(lambda v: 'color:green' if v=='running' else ('color:orange' if v=='idle' else 'color:red'), subset=['state'])

## Workflows

In [ ]:
df_wf = pd.read_sql('SELECT name, source, status, trigger_type, last_run_at, run_count FROM workflows ORDER BY source, name', conn)
if df_wf.empty:
    print('Aucun workflow sauvegardé — exécuter sync_workflows.py pour importer depuis n8n')
else:
    display(df_wf)

## Sync manuel

In [ ]:
# Relancer le snapshot
result = subprocess.run(['python3', str(Path.home()/'IA/Core/jarvis/scripts/sync_config.py'), 'once'],
                        capture_output=True, text=True)
print(result.stdout or result.stderr)
conn = sqlite3.connect(DB)  # reconnect après sync
conn.row_factory = sqlite3.Row
print('Sync OK — relancer les cellules ci-dessus pour rafraîchir')

## Tasks récentes

In [ ]:
df_tasks = pd.read_sql("SELECT id, task_type, status, node, duration, created_at FROM tasks ORDER BY created_at DESC LIMIT 20", conn)
display(df_tasks)